# D3 — Promotion Effectiveness and Discount ROI Charts

This notebook creates the two official D3 charts:

1. **Figure 4.2.7 — D3.1:** Net Sales and Discount-to-Gross % by Promotion
2. **Figure 4.2.8 — D3.2:** Promo Daily Net Sales vs Matched Non-Promo Daily Net Sales

The SQL export files are expected in:

`C:\Users\tpq11\task3_csv`

Charts will be saved to:

`C:\Users\tpq11\charts`


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

YEAR = 2025

CSV_DIR = Path.cwd() / 'task3_csv'
OUT_DIR = Path.cwd() / 'charts'
OUT_DIR.mkdir(exist_ok=True)

print('CSV directory:', CSV_DIR)
print('Chart directory:', OUT_DIR)


## Figure 4.2.7 — D3.1: Net Sales and Discount-to-Gross % by Promotion

**Chart type:** Combo chart

- Bars = Net Sales (RM)
- Line = Discount-to-Gross (%)

This chart compares campaign sales value with discount intensity. A campaign with high net sales should not automatically be treated as effective if it also requires substantial discounting.


In [ ]:
financial = pd.read_csv(CSV_DIR / 'd3_campaign_financial.csv')

financial['Net Sales'] = pd.to_numeric(
    financial['Net Sales'],
    errors='coerce'
)

financial['Discount to Gross %'] = pd.to_numeric(
    financial['Discount to Gross %'],
    errors='coerce'
)

# Use effectiveness order from the confirmed D3.1 ranking
effectiveness_order = [
    'Back To School Bonanza',
    'Mid Year Super Savers 2025',
    'CNY Prosperity Deals 2025',
    'Ramadan Essentials Promo'
]

financial['Promotion'] = pd.Categorical(
    financial['Promotion'],
    categories=effectiveness_order,
    ordered=True
)

financial = financial.sort_values('Promotion')

fig, ax1 = plt.subplots(figsize=(11, 6))

bars = ax1.bar(
    financial['Promotion'].astype(str),
    financial['Net Sales']
)

ax1.set_title(f'Promotion Net Sales and Discount Intensity ({YEAR})')
ax1.set_xlabel('Promotion')
ax1.set_ylabel('Net Sales (RM)')
ax1.tick_params(axis='x', rotation=20)

# Add net sales labels
for bar, value in zip(bars, financial['Net Sales']):
    ax1.annotate(
        f'RM{value:,.0f}',
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        textcoords='offset points',
        xytext=(0, 5),
        ha='center',
        fontsize=8
    )

ax2 = ax1.twinx()

ax2.plot(
    financial['Promotion'].astype(str),
    financial['Discount to Gross %'],
    marker='o'
)

ax2.set_ylabel('Discount-to-Gross (%)')

for x, value in zip(
    financial['Promotion'].astype(str),
    financial['Discount to Gross %']
):
    ax2.annotate(
        f'{value:.1f}%',
        (x, value),
        textcoords='offset points',
        xytext=(0, 8),
        ha='center',
        fontsize=8
    )

fig.tight_layout()

plt.savefig(
    OUT_DIR / 'D3_Chart1_Net_Sales_vs_Discount_Intensity.png',
    dpi=220,
    bbox_inches='tight'
)

plt.show()


## Figure 4.2.8 — D3.2: Promo Daily Sales vs Matched Non-Promo Baseline

**Chart type:** Grouped bar chart

For each campaign:

- Promo Sales/Day = net sales generated per active promotion day
- Baseline Sales/Day = matched non-promotion daily net sales for the same promoted items

This chart directly shows whether the promotion period outperformed normal sales behaviour for the same items.


In [ ]:
baseline = pd.read_csv(CSV_DIR / 'd3_promo_vs_baseline.csv')

for col in ['Promo Sales per Day', 'Baseline Sales per Day', 'Uplift %']:
    baseline[col] = pd.to_numeric(
        baseline[col],
        errors='coerce'
    )

baseline['Promotion'] = pd.Categorical(
    baseline['Promotion'],
    categories=effectiveness_order,
    ordered=True
)

baseline = baseline.sort_values('Promotion')

x = range(len(baseline))
width = 0.36

fig, ax = plt.subplots(figsize=(11, 6))

bars1 = ax.bar(
    [i - width/2 for i in x],
    baseline['Promo Sales per Day'],
    width,
    label='Promo Sales/Day'
)

bars2 = ax.bar(
    [i + width/2 for i in x],
    baseline['Baseline Sales per Day'],
    width,
    label='Matched Baseline/Day'
)

ax.set_title(f'Promotion Daily Sales vs Matched Non-Promo Baseline ({YEAR})')
ax.set_xlabel('Promotion')
ax.set_ylabel('Daily Net Sales (RM)')
ax.set_xticks(list(x))
ax.set_xticklabels(
    baseline['Promotion'].astype(str),
    rotation=20,
    ha='right'
)
ax.legend()

# Add uplift % above each pair
for i, uplift in enumerate(baseline['Uplift %']):
    pair_top = max(
        baseline.iloc[i]['Promo Sales per Day'],
        baseline.iloc[i]['Baseline Sales per Day']
    )
    ax.annotate(
        f'Uplift {uplift:+.1f}%',
        (i, pair_top),
        textcoords='offset points',
        xytext=(0, 8),
        ha='center',
        fontsize=8
    )

fig.tight_layout()

plt.savefig(
    OUT_DIR / 'D3_Chart2_Promo_vs_Matched_Baseline.png',
    dpi=220,
    bbox_inches='tight'
)

plt.show()


## Final files

After running all cells, the notebook creates:

- `D3_Chart1_Net_Sales_vs_Discount_Intensity.png`
- `D3_Chart2_Promo_vs_Matched_Baseline.png`

Use these as the two official D3 figures in the report.
